# Pertemuan 12
## Reverse Logistics

### Kasus

Inspeksi kualitas kemasan kopi ekspor yang dikembalikan oleh distributor di Eropa setelah pengiriman lintas benua:
- Kemasan rusak akibat transit
- Kemasan lembap / berjamur
- Layak diekspor ulang

---
## 1. Setup: AI Vision dengan Hugging Face

In [1]:
%pip install transformers torch pillow

   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   --------- ------------------------------ 2.6/11.0 MB 13.7 MB/s eta 0:00:01
   ------------------- -------------------- 5.2/11.0 MB 13.4 MB/s eta 0:00:01
   ----------------------------- ---------- 8.1/11.0 MB 13.2 MB/s eta 0:00:01
   ---------------------------------------  10.7/11.0 MB 13.2 MB/s eta 0:00:01
   ---------------------------------------- 11.0/11.0 MB 13.0 MB/s  0:00:00
   ---------------------------------------- 0.0/684.4 kB ? eta -:--:--
   ---------------------------------------- 684.4/684.4 kB 13.9 MB/s  0:00:00
   ---------------------------------------- 0.0/4.0 MB ? eta -:--:--
   -------------------------- ------------- 2.6/4.0 MB 12.8 MB/s eta 0:00:01
   ---------------------------------------- 4.0/4.0 MB 12.6 MB/s  0:00:00
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 13.0 MB/s  0:00:00
   --------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from transformers import pipeline

pipe = pipeline(
    "image-classification",
    model="google/vit-base-patch16-224"
)

print("Model siap digunakan.")

c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\USER\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--google--vit-base-patch16-224. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an

Model siap digunakan.


---
## 2. Klasifikasi Gambar Kemasan

Ganti `path/to/image.jpg` dengan path foto kemasan kopi ekspor Anda.

In [3]:
# Ganti dengan path gambar kemasan kopi ekspor Anda
image_path = "asset1.jpg"

results = pipe(image_path)

print("Hasil klasifikasi:")
for r in results[:3]:
    print(f"  {r['label']}: {r['score']:.2%}")

Hasil klasifikasi:
  packet: 77.17%
  corn: 4.16%
  eggnog: 3.53%


---
## 3. Simulasi Dataset Inspeksi

In [4]:
import pandas as pd
import random

random.seed(42)
kondisi = ['Layak Ekspor Ulang', 'Kemasan Lembap/Jamur', 'Kemasan Rusak Transit']
aksi     = {'Layak Ekspor Ulang': 'Kirim ke distributor Eropa', 'Kemasan Lembap/Jamur': 'Repack & uji ulang kualitas', 'Kemasan Rusak Transit': 'Klaim asuransi & daur ulang'}

inspeksi = pd.DataFrame({
    'id_produk': [f'KP-{i:03d}' for i in range(1, 21)],
    'kondisi':   [random.choice(kondisi) for _ in range(20)],
})
inspeksi['aksi'] = inspeksi['kondisi'].map(aksi)

print(inspeksi['kondisi'].value_counts())
inspeksi

kondisi
Layak Ekspor Ulang       11
Kemasan Rusak Transit     7
Kemasan Lembap/Jamur      2
Name: count, dtype: int64


,id_produk,kondisi,aksi
0,KP-001,Kemasan Rusak Transit,Klaim asuransi & daur ulang
1,KP-002,Layak Ekspor Ulang,Kirim ke distributor Eropa
2,KP-003,Layak Ekspor Ulang,Kirim ke distributor Eropa
3,KP-004,Kemasan Rusak Transit,Klaim asuransi & daur ulang
4,KP-005,Kemasan Lembap/Jamur,Repack & uji ulang kualitas
5,KP-006,Layak Ekspor Ulang,Kirim ke distributor Eropa
6,KP-007,Layak Ekspor Ulang,Kirim ke distributor Eropa
7,KP-008,Layak Ekspor Ulang,Kirim ke distributor Eropa
8,KP-009,Kemasan Rusak Transit,Klaim asuransi & daur ulang
9,KP-010,Layak Ekspor Ulang,Kirim ke distributor Eropa


---
## 4. AI Quality Inspection Advisor (Groq)

In [ ]:
from groq import Groq
from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
client = Groq(api_key=os.environ["GROQ_API_KEY"])

summary = inspeksi['kondisi'].value_counts().to_dict()

prompt = f"""
Hasil inspeksi 20 unit kemasan kopi ekspor yang dikembalikan oleh distributor Eropa:
{summary}

Berikan:
1. Analisis kualitas produk
2. Rekomendasi reverse logistics (repack, klaim asuransi, ekspor ulang)
3. Estimasi kerugian dan cara meminimalkannya
"""

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": prompt}]
)

print(response.choices[0].message.content)

Berikut adalah analisis dan rekomendasi berdasarkan hasil inspeksi 20 unit kemasan kopi ekspor yang dikembalikan oleh distributor Eropa:

**1. Analisis Kualitas Produk**

Hasil inspeksi menunjukkan bahwa:
- 55% (11 unit) dinyatakan "Layak Ekspor Ulang", artinya kemasan masih memenuhi standar dan dapat dikirim kembali ke distributor.
- 35% (7 unit) dinyatakan "Kemasan Rusak Transit", yang berarti kemasan mengalami kerusakan fisik selama pengiriman lintas benua, namun produk di dalamnya berpotensi masih dapat diselamatkan.
- 10% (2 unit) dinyatakan "Kemasan Lembap/Jamur", yang berarti kemasan terkontaminasi kelembapan dan berisiko merusak kualitas kopi di dalamnya.

Analisis ini menunjukkan bahwa mayoritas produk masih dalam kondisi baik, namun proporsi kerusakan akibat transit (35%) cukup signifikan dan perlu menjadi perhatian dalam proses pengiriman ekspor jarak jauh.

**2. Rekomendasi Reverse Logistics**

Berikut adalah rekomendasi untuk reverse logistics:
- **Layak Ekspor Ulang (11 u